# TheLook Lakehouse — Stream Processor

Pipeline: **PostgreSQL → Debezium CDC → Kafka → Spark Structured Streaming → Delta Lake (MinIO)**

Run cells in order:
1. Health checks — wait for all services
2. Bootstrap static tables from PostgreSQL
3. Spark Streaming — read CDC events from Kafka, write Delta Lake

## 1 — Health Checks

In [ ]:
import socket, time

def wait_for_tcp(host, port, label, timeout=120):
    print(f"Waiting for {label} ({host}:{port})...", end="", flush=True)
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            socket.create_connection((host, port), timeout=2).close()
            print(" OK"); return
        except OSError:
            print(".", end="", flush=True); time.sleep(3)
    raise TimeoutError(f"{label} not ready after {timeout}s")

wait_for_tcp("postgres",       5432,  "PostgreSQL")
wait_for_tcp("minio",          9000,  "MinIO")
wait_for_tcp("hive-metastore", 9083,  "Hive Metastore")
wait_for_tcp("spark-master",   7077,  "Spark Master")
wait_for_tcp("kafka",          9092,  "Kafka Broker")
wait_for_tcp("schema-registry", 8081, "Schema Registry")

print("\nAll services ready.")

## 2 — Spark Session

In [ ]:
import os

KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:9092")
SCHEMA_REGISTRY_URL = os.getenv("SCHEMA_REGISTRY_URL", "http://schema-registry:8081")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_KEY = os.getenv("MINIO_ROOT_USER", "minio")
MINIO_SECRET = os.getenv("MINIO_ROOT_PASSWORD", "minio123")
DELTA_BASE = "s3a://lakehouse/staging"
CHECKPOINT_BASE = "s3a://lakehouse/checkpoints"

DB_HOST = os.getenv("POSTGRES_HOST", "postgres")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "thelook")
DB_USER = os.getenv("POSTGRES_USER", "admin")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "admin123")

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TheLookStreaming") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.streaming.backpressure.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready.")

## 3 — Bootstrap Static Tables

In [ ]:
from pyspark.sql import functions as F

def bootstrap_table(source_table, target_table, delta_base):
    """Load static reference table from PostgreSQL into Delta Lake."""
    ts_ms = int(time.time() * 1000)
    df = (
        spark.read.format("jdbc")
        .option("url", f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}")
        .option("dbtable", f"public.{source_table}")
        .option("user", DB_USER)
        .option("password", DB_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .load()
        .withColumn("operation", F.lit("r"))
        .withColumn("event_ts_ms", F.lit(ts_ms).cast("long"))
    )
    (
        df.write.format("delta")
        .mode("overwrite")
        .save(f"{delta_base}/{target_table}")
    )
    print(f"Bootstrapped: {target_table} ({df.count()} rows)")

STATIC_TABLES = {
    "products":     "products",
    "dist_centers": "dist_centers",
}

for src, tgt in STATIC_TABLES.items():
    bootstrap_table(src, tgt, DELTA_BASE)

print("\nStatic tables bootstrapped.")

## 4 — CDC Topic Config

In [ ]:
# Debezium topic prefix is "thelook", tables are in "public" schema
# Topic pattern: {prefix}.{schema}.{table}

CDC_TOPICS = {
    "thelook.public.users":      ("users",      "users"),
    "thelook.public.orders":     ("orders",     "orders"),
    "thelook.public.order_items": ("order_items", "order_items"),
    "thelook.public.events":     ("events",     "events"),
}

print("CDC topics:")
for topic, (_, table) in CDC_TOPICS.items():
    print(f"  {topic} -> {DELTA_BASE}/{table}")

## 5 — Parse Debezium CDC Event

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def parse_debezium(cdf: DataFrame) -> DataFrame:
    """Parse Debezium CDC event, extracting the 'after' payload."""
    return (
        cdf
        .withColumn("key_str", F.expr("cast(key as string)"))
        .withColumn("op", F.col("value.after").getField("op").cast("string"))
        .withColumn("ts_ms", F.col("value.ts_ms").cast("long"))
        .withColumn("id",    F.col("value.after").getField("id").cast("string"))
        .withColumn("data",  F.col("value.after"))
        .drop("key", "value", "key_str")
    )

def parse_orders(cdf):
    return (
        cdf
        .withColumn("user_id",     F.col("data").getField("user_id").cast("string"))
        .withColumn("status",      F.col("data").getField("status").cast("string"))
        .withColumn("order_cost",  F.col("data").getField("order_cost").cast("double"))
        .withColumn("created_at",  F.col("data").getField("created_at").cast("timestamp"))
        .drop("data")
    )

def parse_order_items(cdf):
    return (
        cdf
        .withColumn("order_id",    F.col("data").getField("order_id").cast("string"))
        .withColumn("product_id",  F.col("data").getField("product_id").cast("string"))
        .withColumn("quantity",    F.col("data").getField("quantity").cast("int"))
        .withColumn("sale_price",  F.col("data").getField("sale_price").cast("double"))
        .drop("data")
    )

def parse_events(cdf):
    return (
        cdf
        .withColumn("user_id",     F.col("data").getField("user_id").cast("string"))
        .withColumn("event_type",  F.col("data").getField("event_type").cast("string"))
        .withColumn("session_id",  F.col("data").getField("session_id").cast("string"))
        .withColumn("created_at",  F.col("data").getField("created_at").cast("timestamp"))
        .drop("data")
    )

def parse_users(cdf):
    return (
        cdf
        .withColumn("first_name",  F.col("data").getField("first_name").cast("string"))
        .withColumn("last_name",   F.col("data").getField("last_name").cast("string"))
        .withColumn("email",       F.col("data").getField("email").cast("string"))
        .withColumn("age",         F.col("data").getField("age").cast("int"))
        .withColumn("gender",      F.col("data").getField("gender").cast("string"))
        .withColumn("country",     F.col("data").getField("country").cast("string"))
        .withColumn("traffic_source", F.col("data").getField("traffic_source").cast("string"))
        .drop("data")
    )

print("Parse functions defined.")

## 6 — Start CDC Streams

In [ ]:
def start_stream(topic, table_name, parse_fn):
    """Start a Kafka → Delta Lake stream for a CDC topic."""
    raw = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .option("kafka.schema.registry.url", SCHEMA_REGISTRY_URL)
        .load()
    )

    parsed = parse_debezium(raw)
    parsed = parse_fn(parsed)

    query = (
        parsed.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/{table_name}")
        .option("mergeSchema", "true")
        .trigger(processingTime="30 seconds")
        .start(f"{DELTA_BASE}/{table_name}")
    )
    print(f"Stream started: {topic} -> {table_name}")
    return query

# Map topics to their parse functions
PARSE_MAP = {
    "thelook.public.users":       parse_users,
    "thelook.public.orders":      parse_orders,
    "thelook.public.order_items":  parse_order_items,
    "thelook.public.events":      parse_events,
}

streams = []
for topic, (_, table) in CDC_TOPICS.items():
    q = start_stream(topic, table, PARSE_MAP[topic])
    streams.append(q)

print(f"\n{len(streams)} streams running. Interrupt kernel (■) to stop.")

In [ ]:
# Keep streams alive
try:
    spark.streams.awaitAnyTermination()
finally:
    print("Streams stopped.")